# Notebook 04 — Exploratory Data Analysis

**Input:** `data/features_delta.csv` — one row per game, player1 minus player2 deltas

**Purpose:** Understand which delta features separate wins from losses before modeling.
A positive delta means player 1 was higher/faster on that metric.

**Sections:**
1. Load & profile
2. Delta distribution plots (win vs loss)
3. Correlation heatmap
4. Feature differentiation summary (effect size ranking)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pathlib

pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', palette='muted')

DELTA_PATH = '../data/features_delta.csv'
PLOT_DIR   = '../data/plots/'
pathlib.Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

# Metadata columns — excluded from EDA feature plots
META_COLS = ['result', 'filepath', 'duration_min', 'p1_elo', 'p2_elo', 'p1_civ', 'p2_civ']

print('Setup OK')

## Section 1 — Load & Profile

In [ ]:
df = pd.read_csv(DELTA_PATH)
print(f'Shape: {df.shape}')
print(f'Label balance:\n{df["result"].value_counts().to_string()}')
print(f'\np1_elo range: {df["p1_elo"].min():.0f} – {df["p1_elo"].max():.0f}  (median {df["p1_elo"].median():.0f})')
print(f'p2_elo range: {df["p2_elo"].min():.0f} – {df["p2_elo"].max():.0f}  (median {df["p2_elo"].median():.0f})')

delta_cols = [c for c in df.columns if c.endswith('_delta')]
print(f'\nDelta feature columns: {len(delta_cols)}')

nulls = df[delta_cols].isnull().sum()
print(f'\nNull counts (XGBoost handles natively):')
print(nulls[nulls > 0].to_string() if (nulls > 0).any() else '  None')

## Section 2 — Delta Distribution Plots

For each delta feature, overlay the distribution for player-1-wins (result=1) vs
player-1-loses (result=0). A delta centered right of zero in wins means being higher
on that metric than your opponent predicts winning.

In [ ]:
wins   = df[df['result'] == 1]
losses = df[df['result'] == 0]

n_cols = 4
n_rows = int(np.ceil(len(delta_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(delta_cols):
    ax = axes[i]
    w_vals = wins[col].dropna()
    l_vals = losses[col].dropna()
    ax.hist(l_vals, bins=20, alpha=0.55, color='#e07070', label='P1 Loss', density=True)
    ax.hist(w_vals, bins=20, alpha=0.55, color='#70a8e0', label='P1 Win',  density=True)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    # Strip _delta suffix for cleaner titles
    ax.set_title(col.replace('_delta', ''), fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Delta Feature Distributions: P1 Win vs Loss\n(positive = P1 higher than opponent)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
out = f'{PLOT_DIR}delta_distributions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Section 3 — Correlation Heatmap

Filtered to the **top 25 delta features by effect size** (from Section 4 pre-computation).
Showing all 70+ features produces an unreadable matrix — this focuses on the features
that actually separate wins from losses.

Pairs with |r| > 0.7 are flagged below the chart — these are candidates for dropping
one column before modeling to reduce redundant signal.

In [ ]:
from scipy import stats as scipy_stats

# ── Pre-compute effect sizes to rank features for heatmap filtering ───────────
wins   = df[df['result'] == 1]
losses = df[df['result'] == 0]

effect_sizes = {}
for col in delta_cols:
    w = wins[col].dropna()
    l = losses[col].dropna()
    if len(w) < 5 or len(l) < 5:
        continue
    pooled_std = np.sqrt((w.std()**2 + l.std()**2) / 2)
    d = abs((w.mean() - l.mean()) / pooled_std) if pooled_std > 0 else 0.0
    effect_sizes[col] = d

top25_cols = sorted(effect_sizes, key=effect_sizes.get, reverse=True)[:25]
print(f'Top 25 features by effect size selected for heatmap')

# ── Correlation heatmap — top 25 only ────────────────────────────────────────
corr = df[top25_cols].corr()

# Shorten labels: strip _delta suffix
short_labels = [c.replace('_delta', '') for c in top25_cols]

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.1f',
    cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 7},
    xticklabels=short_labels,
    yticklabels=short_labels,
    ax=ax
)
ax.set_title('Delta Feature Correlation Matrix (Top 25 by Effect Size)', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
out = f'{PLOT_DIR}delta_correlation_heatmap.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

# ── Flag high-correlation pairs ───────────────────────────────────────────────
high_corr = [
    (short_labels[i], short_labels[j], corr.iloc[i, j])
    for i in range(len(top25_cols))
    for j in range(i + 1, len(top25_cols))
    if abs(corr.iloc[i, j]) > 0.7
]
if high_corr:
    print(f'\nHigh-correlation pairs (|r| > 0.7) — consider dropping one:')
    for c1, c2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f'  {c1} ↔ {c2}:  r = {r:.2f}')
else:
    print('\nNo pairs with |r| > 0.7 in top 25 features')

## Section 4 — Feature Differentiation Summary

Rank delta features by effect size (Cohen's d) and Mann-Whitney U p-value.
A positive Cohen's d means the delta is larger in wins — player 1 had a bigger
lead over the opponent on that metric in games they won.

In [ ]:
results = []
for col in delta_cols:
    w = wins[col].dropna()
    l = losses[col].dropna()
    if len(w) < 5 or len(l) < 5:
        continue
    stat, p = stats.mannwhitneyu(w, l, alternative='two-sided')
    pooled_std = np.sqrt((w.std()**2 + l.std()**2) / 2)
    d = (w.mean() - l.mean()) / pooled_std if pooled_std > 0 else 0.0
    results.append({
        'feature':     col.replace('_delta', ''),
        'win_mean':    w.mean(),
        'loss_mean':   l.mean(),
        'p_value':     p,
        'cohens_d':    d,
        'abs_d':       abs(d),
        'significant': p < 0.05
    })

summary = pd.DataFrame(results).sort_values('abs_d', ascending=False)
print('Feature Differentiation Summary (ranked by effect size):')
print(summary[['feature','win_mean','loss_mean','p_value','cohens_d','significant']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

out = f'{PLOT_DIR}delta_differentiation_summary.csv'
summary.to_csv(out, index=False)
print(f'\nSaved → {out}')

top = summary.head(20)
colors = ['#70a8e0' if d > 0 else '#e07070' for d in top['cohens_d']]
fig, ax = plt.subplots(figsize=(10, 0.45 * len(top) + 1.5))
ax.barh(top['feature'][::-1], top['cohens_d'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel("Cohen's d  (positive = larger delta in wins)")
ax.set_title('Top Delta Features by Effect Size (P1 Win vs Loss)', fontweight='bold')
plt.tight_layout()
out2 = f'{PLOT_DIR}delta_effect_sizes.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out2}')